In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler


In [2]:
# 1. Load the dataset
df = pd.read_csv('../data/hour.csv')

print(df.head())

   instant      dteday  season  yr  mnth  hr  holiday  weekday  workingday  \
0        1  2011-01-01       1   0     1   0        0        6           0   
1        2  2011-01-01       1   0     1   1        0        6           0   
2        3  2011-01-01       1   0     1   2        0        6           0   
3        4  2011-01-01       1   0     1   3        0        6           0   
4        5  2011-01-01       1   0     1   4        0        6           0   

   weathersit  temp   atemp   hum  windspeed  casual  registered  cnt  
0           1  0.24  0.2879  0.81        0.0       3          13   16  
1           1  0.22  0.2727  0.80        0.0       8          32   40  
2           1  0.22  0.2727  0.80        0.0       5          27   32  
3           1  0.24  0.2879  0.75        0.0       3          10   13  
4           1  0.24  0.2879  0.75        0.0       0           1    1  


In [3]:


# 2. Drop year, index, and leaky variables
drop_cols = ['instant', 'dteday', 'casual', 'registered', 'yr']
df = df.drop(columns=drop_cols)

# 3. Apply Cyclical Encoding
# We convert the linear time features into 2D coordinates on a unit circle
def encode_cyclical(data, col, max_val):
    data[col + '_sin'] = np.sin(2 * np.pi * data[col] / max_val)
    data[col + '_cos'] = np.cos(2 * np.pi * data[col] / max_val)
    return data.drop(columns=[col])

# hr (0-23), mnth (1-12), weekday (0-6)
df = encode_cyclical(df, 'hr', 24)
df = encode_cyclical(df, 'mnth', 12)
df = encode_cyclical(df, 'weekday', 7)

# 4. Define feature groups for scaling
# season and weathersit (1-4) are ordinal; temp/hum/wind are continuous
scale_features = ['season', 'weathersit']

cont_features = ['temp', 'atemp', 'hum', 'windspeed']

# holiday and workingday are binary (already 0/1)
binary_features = ['holiday', 'workingday']
# The newly created sin/cos features are already between -1 and 1
cyclical_features = [col for col in df.columns if '_sin' in col or '_cos' in col]

# 5. Split into Features and Target
X = df.drop(columns=['cnt'])
y = df['cnt']

# 6. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 7. Final Scaling
# We use MinMaxScaler to bring everything into a consistent range.
# This ensures distance parity for the KNN model.
min_max_scaler = MinMaxScaler()
standard_scaler = StandardScaler()
X_train_final = X_train.copy()
X_test_final = X_test.copy()

# Scale everything that isn't already 0-1 or -1 to 1
# We include cyclical features to map them from [-1, 1] to [0, 1] for uniformity
min_max_cols = scale_features
standard_cols = cont_features
X_train_final[min_max_cols] = min_max_scaler.fit_transform(X_train[min_max_cols])
X_test_final[min_max_cols] = min_max_scaler.transform(X_test[min_max_cols])
X_train_final[standard_cols] = standard_scaler.fit_transform(X_train[standard_cols])
X_test_final[standard_cols] = standard_scaler.transform(X_test[standard_cols])


# 8. Reconstruct and Save
train_export = X_train_final.copy()
train_export['y'] = y_train.values

test_export = X_test_final.copy()
test_export['y'] = y_test.values

train_export.to_csv('../data/bike_preprocessed_train.csv', index=False)
test_export.to_csv('../data/bike_preprocessed_test.csv', index=False)

print("--- Advanced Preprocessing Complete ---")
print(f"Cyclical features created: {cyclical_features}")
print(f"Continuous/Ordinal features scaled: {scale_features}")
print(f"Target: cnt (Total Rentals)")

--- Advanced Preprocessing Complete ---
Cyclical features created: ['hr_sin', 'hr_cos', 'mnth_sin', 'mnth_cos', 'weekday_sin', 'weekday_cos']
Continuous/Ordinal features scaled: ['season', 'weathersit']
Target: cnt (Total Rentals)
